In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"....")
data_path = BASE_DIR / "dirty_data.csv"
df = pd.read_csv(data_path, dtype=str)

MISSING_MARKERS = ["", "NULL", "N/A", "n/a", "unknown", "-1", "nan", "???", "no phone", "00000"]
NUMERIC_COLS = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti"]

df = df.replace(MISSING_MARKERS, np.nan)

for col in NUMERIC_COLS:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace(",", ".", regex=False)
    df[col] = pd.to_numeric(df[col], errors="coerce")  # to co się nie da sparsować -> NaN

df.loc[df["loan_amnt"] < 0, "loan_amnt"] = np.nan
df.loc[(df["fico_score"].astype(float) < 300) | (df["fico_score"].astype(float) > 850), "fico_score"] = np.nan
df.loc[df["dti"] > 100, "dti"] = np.nan
df.loc[df["annual_inc"] < 0, "annual_inc"] = np.nan

df["home_ownership"] = df["home_ownership"].str.strip().str.upper()
df["grade"] = df["grade"].str.strip().str.upper().str.extract(r"([A-G])")  # wyciąga literę A-G nawet z "grade B "

HOME_OWNERSHIP_FIXES = {
    "RENTED": "RENT",
    "MORTGAGED": "MORTGAGE",
}

df['home_ownership'] = df['home_ownership'].replace(HOME_OWNERSHIP_FIXES)

In [ ]:
from rapidfuzz import process, fuzz

CANONICAL_PURPOSE = [
    "debt_consolidation", "credit_card", "home_improvement", "major_purchase",
    "small_business", "car", "medical", "other"
]

def normalize_purpose(val):
    if pd.isna(val):
        return np.nan
    cleaned = val.strip().lower().replace(" ", "_").replace("-", "_")
    match, score, _ = process.extractOne(cleaned, CANONICAL_PURPOSE, scorer=fuzz.ratio)
    return match if score >= 80 else np.nan

df["purpose_original"] = df["purpose"]              # kopia surowych danych
df["purpose"] = df["purpose"].apply(normalize_purpose)   # jedno czyszczenie

podejrzane = df[df["purpose"].isna() & df["purpose_original"].notna()]
print(len(podejrzane))
print(podejrzane["purpose_original"].unique())

0
[]


In [3]:
def normalize_phone(p):
    if pd.isna(p):
        return np.nan
    digits = "".join(ch for ch in str(p) if ch.isdigit())
    if len(digits) != 10:
        return np.nan
    return f"{digits[0:3]}-{digits[3:6]}-{digits[6:10]}"

przed = df["phone"].isna().sum()
df["phone"] = df["phone"].apply(normalize_phone)
po = df["phone"].isna().sum()
print(f"braki przed: {przed}, po: {po}, nowo dodane NaN: {po - przed}")

braki przed: 3037, po: 3037, nowo dodane NaN: 0


In [4]:
for col in ["zip_code", "city", "state", "emp_length", "verification_status", "loan_status"]:
    print(col, "->", df[col].unique()[:15])

zip_code -> ['94050' '99700' '99818' '86174' '75977' '33529' '95830' '61735' '37148'
 '10784' '02065' '08876' '23456' '12879' '48376']
city -> ['Welchland' 'South Laurenside' 'Mitchellside' 'North Denise' 'Boydmouth'
 'Lake Danielle' 'North Julieside' 'North Morganshire' 'Lake Anne'
 'Gabrielport' 'Benjaminville' 'Lake Heatherfort' 'East Dana'
 'Katherineberg' 'South Rebecca']
state -> ['OH' 'NH' 'TX' 'CA' 'SC' 'MO' 'FM' 'ME' 'VI' 'IA' 'ND' 'RI' 'NE' 'AZ'
 'NY']
emp_length -> ['8 years' '3 years' '7 years' '10+ years' '2 years' '< 1 year' '1 year'
 '4 years' '6 years' '5 years' '9 years']
verification_status -> ['Verified' 'Not Verified' 'Source Verified']
loan_status -> ['Charged Off' 'Current' 'Fully Paid' 'Late (31-120 days)']


In [ ]:
df.to_csv(BASE_DIR / "layer1_cleaned.csv", index=False)

In [ ]:
import recordlinkage

df_full = pd.read_csv(data_path, dtype=str)  # potrzebujemy full_name, email, phone, street_address - są w oryginalnym pliku
df_full["last_name"] = df_full["full_name"].str.strip().str.split().str[-1].str.upper()

indexer = recordlinkage.Index()
indexer.block(["last_name", "zip_code"])
candidate_pairs = indexer.index(df_full)

compare = recordlinkage.Compare()
compare.string("full_name", "full_name", method="jarowinkler", label="name_score")
compare.string("email", "email", method="jarowinkler", label="email_score")
compare.exact("phone", "phone", label="phone_score")
compare.string("street_address", "street_address", method="jarowinkler", label="address_score")
features = compare.compute(candidate_pairs, df_full)
features["total_score"] = features[["name_score","email_score","phone_score","address_score"]].sum(axis=1)

def pair_to_ids(idx_pair):
    return df_full.loc[idx_pair[0], "customer_id"], df_full.loc[idx_pair[1], "customer_id"]
features["id_a"], features["id_b"] = zip(*[pair_to_ids(p) for p in features.index])

LOW, HIGH = 2.3, 2.6
gray_zone = features[(features["total_score"] >= LOW) & (features["total_score"] < HIGH)]
gray_zone[["id_a", "id_b", "total_score"]].to_csv(BASE_DIR / "duplicate_candidates_gray.csv", index=False)
print(f"Strefa szara: {len(gray_zone)} par -> duplicate_candidates_gray.csv")

auto_dup = features[features["total_score"] >= HIGH]
auto_not = features[features["total_score"] < LOW]

auto_dup[["id_a", "id_b", "total_score"]].to_csv(BASE_DIR / "auto_duplicate.csv", index=False)
auto_not[["id_a", "id_b", "total_score"]].to_csv(BASE_DIR / "auto_not_duplicate.csv", index=False)

print(f"AUTO-duplikat: {len(auto_dup)}, AUTO-nie-duplikat: {len(auto_not)}")

Strefa szara: 17 par -> duplicate_candidates_gray.csv


In [ ]:
import asyncio
import json
from typing import Optional
from pydantic import BaseModel
from google import genai
import time

class RateLimiter:
    def __init__(self, calls_per_minute: int):
        self.min_interval = 60 / calls_per_minute  # np. 60/15 = 4 sekundy między startami
        self.last_call = 0
        self.lock = asyncio.Lock()

    async def wait(self):
        async with self.lock:
            now = time.monotonic()
            elapsed = now - self.last_call
            if elapsed < self.min_interval:
                await asyncio.sleep(self.min_interval - elapsed)
            self.last_call = time.monotonic()

# rate_limiter = RateLimiter(calls_per_minute=14)  # 14 zamiast 15 - mały margines bezpieczeństwa
rate_limiter = RateLimiter(calls_per_minute=100)

GEMINI_API_KEY = "API_KEY"
MODEL_NAME = "gemini-3.5-flash-lite"
CHUNK_SIZE = 100
MAX_CONCURRENT = 10
MAX_RETRIES = 3

class CleanedDateNotes(BaseModel):
    customer_id: str
    application_date: Optional[str] = None
    issue_date: Optional[str] = None
    advisor_notes: Optional[str] = None

class CleanedChunk(BaseModel):
    records: list[CleanedDateNotes]

DATE_NOTES_SCHEMA = """
Jesteś narzędziem do czyszczenia danych bankowych. Otrzymujesz rekordy zawierające
TYLKO: customer_id, application_date, issue_date, advisor_notes.

- daty -> format "YYYY-MM-DD". Wejście może być w różnych formatach (US m/d/y,
  europejski d/m/y, ISO, nazwa miesiąca słownie lub skrótem). issue_date jest
  zawsze 1-30 dni PO application_date - użyj tego do rozstrzygania niejednoznacznych
  przypadków (np. 07/11/2015 - 7 listopada czy 11 lipca?). Data fizycznie
  niemożliwa (np. 31.02) -> null.
- advisor_notes -> popraw literówki, usuń nadmiarowe białe znaki, normalna
  wielkość liter (nie CAPS LOCK), zachowaj oryginalny sens i długość zdania.

Zwróć WSZYSTKIE rekordy z paczki, w tej samej kolejności, customer_id niezmienione.
"""

def chunk_list(data: list, chunk_size: int = CHUNK_SIZE):
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

async def clean_chunk(chunk: list[dict], semaphore: asyncio.Semaphore,
                       client: genai.Client, chunk_num: int) -> list[dict]:
    async with semaphore:
        await rate_limiter.wait()
        prompt = DATE_NOTES_SCHEMA + "\n\nOczyść poniższe rekordy:\n" + json.dumps(chunk, default=str)

        for attempt in range(MAX_RETRIES):
            try:
                response = await client.aio.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config={"response_mime_type": "application/json",
                            "response_schema": CleanedChunk},
                )
                parsed = CleanedChunk.model_validate_json(response.text)

                if len(parsed.records) != len(chunk):
                    raise ValueError(f"dostałem {len(chunk)}, wróciło {len(parsed.records)}")

                print(f"  paczka {chunk_num}: OK ({len(parsed.records)} rekordów)")
                return [r.model_dump() for r in parsed.records]

            except Exception as e:
                wait = 2 ** attempt
                print(f"  paczka {chunk_num}: błąd (próba {attempt+1}/{MAX_RETRIES}): {e} -> czekam {wait}s")
                await asyncio.sleep(wait)

        print(f"  paczka {chunk_num}: POMINIĘTA po {MAX_RETRIES} próbach")
        return []

async def resolve_gray_zone_duplicates(pairs_df: pd.DataFrame, df_full: pd.DataFrame,
                                        client: genai.Client) -> pd.DataFrame:
    DUPLICATE_SCHEMA = """
    Dla każdej pary rekordów klientów oceń, czy to TEN SAM klient wprowadzony do
    systemu dwa razy (duplikat, np. z literówką/inną wielkością liter w danych),
    czy DWIE RÓŻNE osoby z podobnymi danymi (to samo nazwisko i zip, ale różny
    email/telefon/adres). Zwróć is_duplicate: true/false dla każdej pary.
    """

    class DuplicateVerdict(BaseModel):
        record_a_id: str
        record_b_id: str
        is_duplicate: bool

    class DuplicateVerdicts(BaseModel):
        verdicts: list[DuplicateVerdict]

    fields = ["customer_id", "full_name", "email", "phone", "street_address", "zip_code"]
    payload = []

    for _, row in pairs_df.iterrows():
        rec_a = df_full.loc[df_full["customer_id"] == row["id_a"], fields].iloc[0].to_dict()
        rec_b = df_full.loc[df_full["customer_id"] == row["id_b"], fields].iloc[0].to_dict()
        payload.append({"record_a": rec_a, "record_b": rec_b})

    prompt = DUPLICATE_SCHEMA + "\n\nPary do oceny:\n" + json.dumps(payload, default=str)
    response = await client.aio.models.generate_content(
        model=MODEL_NAME, contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": DuplicateVerdicts},
    )
    parsed = DuplicateVerdicts.model_validate_json(response.text)
    return pd.DataFrame([v.model_dump() for v in parsed.verdicts])


async def main():
    df = pd.read_csv(BASE_DIR / "layer1_cleaned.csv", dtype=str)
    OUTPUT_PATH = BASE_DIR / "layer2_dates_notes_full.csv"

    # sprawdź co już zostało oczyszczone w poprzednich dniach
    if OUTPUT_PATH.exists():
        already_done = pd.read_csv(OUTPUT_PATH)
        done_ids = set(already_done["customer_id"])
        print(f"Już oczyszczone wcześniej: {len(done_ids)} rekordów")
    else:
        already_done = pd.DataFrame()
        done_ids = set()

    # zostają tylko te, których jeszcze nie ma
    todo = df[~df["customer_id"].isin(done_ids)]
    records = todo[["customer_id", "application_date", "issue_date", "advisor_notes"]].to_dict(orient="records")
    print(f"Do zrobienia dzisiaj: {len(records)} rekordów ({len(records)//CHUNK_SIZE + 1} paczek)")

    chunks = list(chunk_list(records, CHUNK_SIZE))

    client = genai.Client(api_key=GEMINI_API_KEY)
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    tasks = [clean_chunk(chunk, semaphore, client, i + 1) for i, chunk in enumerate(chunks)]
    results = await asyncio.gather(*tasks)
    new_cleaned = [record for chunk_result in results for record in chunk_result]

    # DOPISZ do tego co już było, nie nadpisuj
    combined = pd.concat([already_done, pd.DataFrame(new_cleaned)], ignore_index=True)
    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"Zapisano łącznie: {len(combined)} / {len(df)} rekordów")

await main()
# w jupyter: await main()
# jako plik .py: asyncio.run(main())

Już oczyszczone wcześniej: 38100 rekordów
Do zrobienia dzisiaj: 63400 rekordów (635 paczek)


Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


  paczka 2: OK (100 rekordów)
  paczka 6: OK (100 rekordów)
  paczka 4: OK (100 rekordów)
  paczka 1: OK (100 rekordów)
  paczka 9: OK (100 rekordów)
  paczka 3: OK (100 rekordów)
  paczka 5: OK (100 rekordów)
  paczka 7: OK (100 rekordów)
  paczka 8: OK (100 rekordów)
  paczka 10: OK (100 rekordów)
  paczka 11: OK (100 rekordów)
  paczka 15: OK (100 rekordów)
  paczka 16: OK (100 rekordów)
  paczka 12: OK (100 rekordów)
  paczka 17: OK (100 rekordów)
  paczka 13: OK (100 rekordów)
  paczka 18: OK (100 rekordów)
  paczka 14: OK (100 rekordów)
  paczka 19: OK (100 rekordów)
  paczka 20: OK (100 rekordów)
  paczka 21: OK (100 rekordów)
  paczka 22: OK (100 rekordów)
  paczka 25: OK (100 rekordów)
  paczka 23: OK (100 rekordów)
  paczka 28: OK (100 rekordów)
  paczka 24: OK (100 rekordów)
  paczka 26: OK (100 rekordów)
  paczka 27: OK (100 rekordów)
  paczka 30: OK (100 rekordów)
  paczka 29: OK (100 rekordów)
  paczka 31: OK (100 rekordów)
  paczka 34: OK (100 rekordów)
  paczka 32: OK (

In [ ]:
cleaned = pd.read_csv(BASE_DIR / "layer2_dates_notes_full.csv")
truth = pd.read_csv(BASE_DIR / "clean_data.csv")

merged = cleaned.merge(truth, on="customer_id", suffixes=("_llm", "_truth"))
print(f"Dopasowanych rekordów: {len(merged)} / {len(cleaned)} w cleaned")

for col in ["application_date", "issue_date"]:
    correct = (pd.to_datetime(merged[f"{col}_llm"], errors="coerce") == pd.to_datetime(merged[f"{col}_truth"], errors="coerce")).sum()
    print(f"{col}: {correct}/{len(merged)} poprawnie ({correct/len(merged)*100:.2f}%)")

Dopasowanych rekordów: 100000 / 101500 w cleaned
application_date: 98784/100000 poprawnie (98.78%)
issue_date: 99524/100000 poprawnie (99.52%)


In [21]:
zle_daty = merged[pd.to_datetime(merged["application_date_llm"], errors="coerce") != 
                   pd.to_datetime(merged["application_date_truth"], errors="coerce")]

print(f"Błędnych: {len(zle_daty)}")
print(zle_daty[["customer_id", "application_date_llm", "application_date_truth"]].sample(15, random_state=1).to_string())

Błędnych: 442
      customer_id application_date_llm application_date_truth
22875    LC146679           2020-07-10             2020-10-07
36832    LC162108           2021-02-06             2021-06-02
26939    LC122548           2017-09-11             2017-11-09
4404     LC154248           2018-09-12             2018-12-09
31823    LC115341           2023-02-09             2023-09-02
37272    LC116160           2019-03-12             2019-12-03
26504    LC144501           2015-03-01             2015-01-03
37299    LC100955           2022-12-10             2022-10-12
26107    LC108815           2022-04-07             2022-07-04
902      LC134988           2016-02-02             2016-02-08
37335    LC130891           2022-09-11             2022-11-09
26112    LC171237           2016-06-08             2016-08-06
18766    LC104573           2018-05-11             2018-11-05
34627    LC135720           2019-05-09             2019-09-05
16272    LC179650           2023-05-12             2023-

In [22]:
def is_swap(llm, truth):
    try:
        y1, m1, d1 = llm.split("-")
        y2, m2, d2 = truth.split("-")
        return y1 == y2 and m1 == d2 and d1 == m2 and m1 != d1  # wyklucz np. 05-05 (nie do odróżnienia)
    except:
        return False

zle_daty["is_swap"] = zle_daty.apply(
    lambda r: is_swap(r["application_date_llm"], r["application_date_truth"]), axis=1
)
print(f"{zle_daty['is_swap'].sum()} / {len(zle_daty)} błędów to zamiana dzień<->miesiąc")

403 / 442 błędów to zamiana dzień<->miesiąc


In [23]:
inne_bledy = zle_daty[~zle_daty["is_swap"]]
print(f"Liczba: {len(inne_bledy)}")
print(inne_bledy[["customer_id", "application_date_llm", "application_date_truth"]].to_string())

Liczba: 39
      customer_id application_date_llm application_date_truth
322      LC145945                  NaN             2015-04-04
902      LC134988           2016-02-02             2016-02-08
1018     LC197665           2022-04-04             2022-04-05
2585     LC144347           2019-08-08             2019-08-09
2632     LC137701           2019-08-08             2019-08-04
6222     LC153840           2022-01-01             2022-01-10
6357     LC188443           2018-01-01             2018-01-11
6603     LC191352                  NaN             2015-06-01
7936     LC142085           2023-04-04             2023-04-08
8012     LC120435           2023-04-04             2023-04-05
9936     LC103508           2023-07-07             2023-07-08
10942    LC159691           2015-06-21             2015-03-31
10981    LC169206           2018-06-06             2018-06-10
11697    LC130024           2018-10-10             2018-10-01
12136    LC180633                  NaN             2023-11-

In [ ]:
"""
merge_final.py
Scala wszystkie warstwy czyszczenia w jeden finalny, gotowy plik:
  - layer1_cleaned.csv           (kod: wszystko poza datami/notatkami)
  - layer2_dates_notes_full.csv  (LLM: application_date, issue_date, advisor_notes)
  - auto_duplicate.csv           (kod: pary pewnych duplikatów)
  - duplicate_verdicts_llm.csv   (LLM: rozstrzygnięcie strefy szarej)
 
Wynik: final_cleaned_data.csv - kompletny, oczyszczony zbiór z flagą duplikatów.
"""
 
# ============================================================
# 1. WCZYTANIE WSZYSTKICH WARSTW
# ============================================================
 
layer1 = pd.read_csv(BASE_DIR / "layer1_cleaned.csv", dtype=str)
layer2 = pd.read_csv(BASE_DIR / "layer2_dates_notes_full.csv", dtype=str)
auto_dup = pd.read_csv(BASE_DIR / "auto_duplicate.csv", dtype=str)
llm_verdicts = pd.read_csv(BASE_DIR / "duplicate_verdicts_llm.csv", dtype=str)
 
print(f"layer1: {len(layer1)} wierszy")
print(f"layer2: {len(layer2)} wierszy")
 
# ============================================================
# 2. PODMIANA KOLUMN DAT/NOTATEK - z layer1 (kod) na layer2 (LLM)
# ============================================================
 
# usuwamy z layer1 kolumny, które LLM czyścił, żeby nie było duplikatów po merge
llm_cols = ["application_date", "issue_date", "advisor_notes"]
layer1_bez_llm_cols = layer1.drop(columns=llm_cols)
 
final = layer1_bez_llm_cols.merge(layer2, on="customer_id", how="left")
 
print(f"Po scaleniu layer1 + layer2: {len(final)} wierszy")
 
# ============================================================
# 3. FLAGA DUPLIKATÓW - łączymy decyzje kodu i LLM w jedną kolumnę
# ============================================================
 
final["is_duplicate"] = False
final["duplicate_of"] = None
 
# 3a. Pary, które kod uznał za pewne duplikaty (próg >= HIGH)
for _, row in auto_dup.iterrows():
    final.loc[final["customer_id"] == row["id_a"], "is_duplicate"] = True
    final.loc[final["customer_id"] == row["id_a"], "duplicate_of"] = row["id_b"]
 
# 3b. Pary ze strefy szarej, które LLM ocenił jako duplikat (is_duplicate == "True")
llm_verdicts["is_duplicate"] = llm_verdicts["is_duplicate"].astype(str).str.lower() == "true"
confirmed_by_llm = llm_verdicts[llm_verdicts["is_duplicate"]]
 
for _, row in confirmed_by_llm.iterrows():
    final.loc[final["customer_id"] == row["record_a_id"], "is_duplicate"] = True
    final.loc[final["customer_id"] == row["record_a_id"], "duplicate_of"] = row["record_b_id"]
 
n_duplicates = final["is_duplicate"].sum()
print(f"Oznaczonych jako duplikat: {n_duplicates}")
 
# ============================================================
# 4. ZAPIS FINALNEGO PLIKU
# ============================================================
 
OUTPUT = BASE_DIR / "final_cleaned_data.csv"
final.to_csv(OUTPUT, index=False)
print(f"\nZapisano finalny plik: {OUTPUT}")
print(f"Wiersze: {len(final)}, kolumny: {len(final.columns)}")
print(f"\nKolumny: {list(final.columns)}")


layer1: 101500 wierszy
layer2: 101500 wierszy
Po scaleniu layer1 + layer2: 101500 wierszy
Oznaczonych jako duplikat: 1473

Zapisano finalny plik: C:\Users\Łukasz\OneDrive\Dokumenty\github portfolio\ETL with LLM\final_cleaned_data.csv
Wiersze: 101500, kolumny: 28

Kolumny: ['customer_id', 'full_name', 'email', 'phone', 'street_address', 'city', 'state', 'zip_code', 'loan_amnt', 'term_months', 'int_rate', 'installment', 'grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'fico_score', 'loan_status', 'purpose_original', 'application_date', 'issue_date', 'advisor_notes', 'is_duplicate', 'duplicate_of']


In [13]:
print(final["is_duplicate"].sum())
print(final[final["is_duplicate"]]["customer_id"].duplicated().sum())  # czy są powtórzenia

1473
0
